<a href="https://colab.research.google.com/github/Marco967a/Bibliothek/blob/main/IRIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook IRIS

## SCOPO

Lo scopo di questo notebook è effettuare la data cleaing, EDA e normalizzazione con tre differenti scaler

---

## 1. Importazione delle Librerie e caricamento dataset

### 1.1 Importazione delle librerie

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
my_palette = [
    "#377eb8", "#e41a1c", "#4daf4a", "#984ea3", "#ff7f00",
    "#ffff33", "#a65628", "#f781bf", "#999999"
]
custom_params = {"axes.edgecolor": "black", "lines.linewidth": 2.5}
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.2, rc=custom_params, palette=my_palette)

from scipy import stats
from scipy.stats import skew, kurtosis, chi2_contingency

### 1.2 Caricamento del Dataset

In [ ]:
import os

DATA_FILE = 'iris.csv'
LOCAL_PATH = os.path.join('data', DATA_FILE)
REMOTE_URL = f'https://raw.githubusercontent.com/Marco967a/Portfolio_Notebooks/main/data/{DATA_FILE}'

# Caricamento con fallback: locale se presente, altrimenti GitHub Raw (per Google Colab)
data_path = LOCAL_PATH if os.path.exists(LOCAL_PATH) else REMOTE_URL
df = pd.read_csv(data_path)
if df is None or df.empty:
    raise ValueError('Errore nel caricamento del dataset')


Mounted at /content/drive


## 2. Esplorazione Iniziale della Struttura

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

## 3. Informazioni Generali sul Dataset

In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
df = df.drop(columns=['Id'])

### Osservazioni

Abbiamo notato le dimensioni del dataset e osservato i valori iniziali e finali. Il totale di valori unici per feature. Di ogni colonna, abbiamo identificato il suo tipo.

---

## 4. Pulizia dati

### 4.1 Rimozione dei duplicati

In [ ]:
df_duplicates = df[df.duplicated()]
df_duplicates

In [ ]:
df.drop_duplicates(inplace=True)

### 4.2 Valori mancanti

In [ ]:
sns.heatmap(df.isna(),cbar=False)

In [ ]:
mancanti = df.isnull().sum()
mancanti

In [ ]:
df.shape

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

In [ ]:
print(categorical_cols)
print(f"\nTotale: {len(categorical_cols)} feauters\n")

In [ ]:
dfca = df[categorical_cols]
dfca.describe(include='all')

Abbiamo controllato i valori unici per ogni feature e la frequenza della moda

In [ ]:
for col in categorical_cols:
    print(f"\n--- {col.upper()} ---")
    value_counts = df[col].value_counts()
    print(value_counts)
    print(f"\nNumero di categorie uniche: {df[col].nunique()}")
    print("=" * 60)

### 4.3 Gestione outliers

In [ ]:
outlier = {}

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    sotto = Q1 - 1.5 * IQR
    sopra = Q3 + 1.5 * IQR

    outliers = df[(df[col] < sotto) | (df[col] > sopra)]
    n_outliers = len(outliers)
    percentuale = (n_outliers / len(df)) * 100

    outlier = {
        'N. Outliers': n_outliers,
        'Percentuale': percentuale,
        'Baffo inferiore': sotto,
        'Baffo superiore': sopra
    }

    print(f"{col}:")
    print(f"  Limiti: [{sotto:.2f}, {sopra:.2f}]")
    print(f"  Outliers: {n_outliers:,} ({percentuale:.2f}%)")


## 5. EDA

In [ ]:
print(numerical_cols)
print(f"\nTotale: {len(numerical_cols)} feauters\n")

In [ ]:
for col in numerical_cols:
    print(f"\n{'='*60}")
    print(f"Variabile: {col.upper()}")
    print(f"{'='*60}")

    stats_dict = {
        'Count': df[col].count(),
        'Media': df[col].mean(),
        'Mediana': df[col].median(),
        'Moda': df[col].mode()[0] if len(df[col].mode()) > 0 else np.nan,
        'Dev. Standard': df[col].std(),
        'Coefficiente di Variazione': df[col].std() / df[col].mean() if df[col].mean() != 0 else np.nan,
        'Minimo': df[col].min(),
        'Massimo': df[col].max(),
        'Range': df[col].max() - df[col].min(),
        'Q1 (25%)': df[col].quantile(0.25),
        'Q3 (75%)': df[col].quantile(0.75),
        'IQR': df[col].quantile(0.75) - df[col].quantile(0.25),
        'Skewness': df[col].skew(),
        'Kurtosis': df[col].kurtosis()
    }

    for key, value in stats_dict.items():
        print(f"{key:30s}: {value:,.2f}")

In [ ]:
for col in numerical_cols:

    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    sns.histplot(df[col], kde=True)
    plt.axvline(x=df[col].mean(), color='red', linestyle='-', linewidth=2, label='Media')
    plt.axvline(x=df[col].median(), color='blue', linestyle='--', linewidth=2, label='Mediana')

    plt.title(f'Istogramma: {col}')
    plt.xlabel(col)
    plt.ylabel('Frequenza')
    plt.legend()

    plt.subplot(1, 3, 2)
    sns.boxplot(y=df[col])
    plt.title(f'Box Plot: {col}')
    plt.ylabel('')

    plt.subplot(1, 3, 3)
    stats.probplot(df[col], dist="norm", plot=plt)
    plt.title(f'QQ Plot per la Normalità: {col}')
    plt.show()

    plt.show()

## Resoconto del Contenuto del Dataset

### Descrizione Generale

**Dimensioni del dataset:**
- **Righe**: 147
- **Colonne**: 5

---

### Descrizione delle Variabili

| Colonna | Tipo | Descrizione |
|---------|------|-------------|
| **SepalLengthCm** | Numerica | Lunghezza in cm dei sepali |
| **SepalWidthCm** | Numerica | Larghezza in cm dei sepali |
| **PetalLengthCm** | Numerica | Lunghezza dei petali in cm dei sepali |
| **PetalWidthCm** | Numerica | Larghezza dei petali in cm dei sepali |


#### Variabile Target

| Colonna | Tipo | Descrizione |
|---------|------|-------------|
| **Species** | Categorica | Specie identificativa dell'iris |


---

### Osservazioni
Il dataset è pulito, non presenta alcun valore mancante. Ad eccezione di qualche duplicato che p stato rimosso.
  
Non è stato necessario formattare il testo delle modalità presenti poiché le feautere risultano numeriche ad eccezione della viarbilie target che presenta tre modalità.

#### Outliers  

Sono stati rilevati 4 outlier con il metodo della diffeeranza interquartilistica.

---

In [ ]:
sns.pairplot(df, hue='Species', diag_kind='kde')

In [ ]:
matrice = df[numerical_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(matrice, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Matrice di Correlazione - Variabili Numeriche', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("\nCorrelazioni Forti (|r| > 0.5):\n")
for i in range(len(matrice.columns)):
    for j in range(i+1, len(matrice.columns)):
        if abs(matrice.iloc[i, j]) > 0.5:
            print(f"{matrice.columns[i]:20s} <-> {matrice.columns[j]:20s}: {matrice.iloc[i, j]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    if idx < len(axes):
        sns.violinplot(x='Species', y=df[col], data=df, ax=axes[idx], palette=my_palette)
        axes[idx].set_title(f'{col} per Species', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Species', fontsize=10)
        axes[idx].set_ylabel(col, fontsize=10)

plt.suptitle('Distribuzione Variabili Numeriche per Classe Species',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Dalla comparazione delle lunghezze e larghezze di petali e sepali, notiamo che:

- Iris setosa ha una lunghezza del sepalo più corta delle altre due specie, non comparabile con le altre. La larghezza del sepalo è invece la più ampia rispetto le altre specie.
- Iris virginica ha la lunghezza del sepalo più lunga, anche se alcuni esemplari di Iris versicolor hanno lunghezze del sepalo equiparabili.
- La largehzza del sepalo delle due specie Iris versicolo e Irisi virginica sono equiparabili, anche se più frequentemente il sepalo nella specie Irisvirginica tende ad essere più largo.
- I petali di Iris setosa non sono confrontabili con le altre specie che sono più lunghe, l'Iris versicolo ha un lunghezza del petalo generalemnte più corta della Iris virginica. Allo stesso modo questo si riflette sulla larghezza del petalo.

In [ ]:
target_column = 'Species'
X = df[numerical_cols]

## 6. Normalizzazione

### 6.1 Min-Max

In [ ]:
# 1. MIN-MAX SCALING
minmax_scaler = MinMaxScaler(feature_range=(0, 1))
X_minmax = minmax_scaler.fit_transform(X)
df_minmax = pd.DataFrame(X_minmax, columns=numerical_cols)

In [ ]:
df_minmax.head()

In [ ]:
df_minmax.describe()

### 6.2 Standard scaling

In [ ]:
# 2. STANDARD SCALING (Z-score normalization)
standard_scaler = StandardScaler()
X_standard = standard_scaler.fit_transform(X)
df_standard = pd.DataFrame(X_standard, columns=numerical_cols)

In [ ]:
df_standard.head()

In [ ]:
df_standard.describe()

### 6.3 Robust scaling

In [ ]:
# 3. ROBUST SCALING
robust_scaler = RobustScaler()
X_robust = robust_scaler.fit_transform(X)
df_robust = pd.DataFrame(X_robust, columns=numerical_cols)

In [ ]:
df_robust.head()

In [ ]:
df_robust.describe()

In [ ]:
for feature in numerical_cols:
    fig, axes = plt.subplots(1, 4, figsize=(24, 5))
    fig.suptitle(f'Confronto Completo - {feature}', fontsize=16, fontweight='bold')

    # Dati originali
    sns.histplot(data=df, x=feature, kde=True, ax=axes[0], color='gray', stat='count')
    axes[0].set_title('Dati Originali', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Valore')
    axes[0].set_ylabel('Densità')
    skew_original = df[feature].skew()
    axes[0].text(0.95, 0.95, f'Skewness: {skew_original:.3f}',
                 transform=axes[0].transAxes, fontsize=10,
                 verticalalignment='top', horizontalalignment='right',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # MinMaxScaler
    sns.histplot(data=df_minmax, x=feature, kde=True, ax=axes[1], color='skyblue', stat='count')
    axes[1].set_title('MinMax Scaling', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Valore Normalizzato')
    axes[1].set_ylabel('')
    skew_minmax = df_minmax[feature].skew()
    axes[1].text(0.95, 0.95, f'Skewness: {skew_minmax:.3f}',
                 transform=axes[1].transAxes, fontsize=10,
                 verticalalignment='top', horizontalalignment='right',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # StandardScaler
    sns.histplot(data=df_standard, x=feature, kde=True, ax=axes[2], color='lightcoral', stat='count')
    axes[2].set_title('Standard Scaling', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Valore Normalizzato')
    axes[2].set_ylabel('')
    skew_standard = df_standard[feature].skew()
    axes[2].text(0.95, 0.95, f'Skewness: {skew_standard:.3f}',
                 transform=axes[2].transAxes, fontsize=10,
                 verticalalignment='top', horizontalalignment='right',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # RobustScaler
    sns.histplot(data=df_robust, x=feature, kde=True, ax=axes[3], color='lightgreen', stat='count')
    axes[3].set_title('Robust Scaling', fontsize=12, fontweight='bold')
    axes[3].set_xlabel('Valore Normalizzato')
    axes[3].set_ylabel('')
    skew_robust = df_robust[feature].skew()
    axes[3].text(0.95, 0.95, f'Skewness: {skew_robust:.3f}',
                 transform=axes[3].transAxes, fontsize=10,
                 verticalalignment='top', horizontalalignment='right',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    plt.show()


### Osservazioni sulla normalizzazione

Dall'analisi visiva degli istogrammi per ogni feature emerge che:

1. La forma della distribuzione rimane invariata, tutte e tre le tecniche di normalizzazione preservano la forma originale della distribuzione dei dati. Questo è confermato dal fatto che i valori di skewness rimangono identici tra i diversi metodi di scaling.

2. Cambiano solo scala e posizione:
  * MinMaxScaler: comprime i valori nell'intervallo [0,1]
  * StandardScaler: centra i dati su media=0 e deviazione standard=1
  * RobustScaler: centra sulla mediana utilizzando l'IQR come scala

Lo scopo ultimo è sempre quello di mantenere le informazioni della distribuzione ma rendendo i valori interni comprensibili e meglio gestibili per gli algoritmi di Machine Learning.
